# 📊 Workshop: Quantitatively Evaluating DriveVLM-T5 with COCO Captioning Metrics

## 🎯 Learning Objectives
Upon completion of this notebook, you will be able to:
- **Load** a trained DriveVLM-T5 checkpoint (produced by `train_T5_Base.ipynb`) for inference
- **Generate** free-form text answers with `model.generate()` (as opposed to teacher-forced training)
- **Evaluate** using standard image-captioning metrics (BLEU, METEOR, ROUGE-L, CIDEr) to score generated driving explanations against ground-truth answers

## 🗺️ What This Notebook Does
Training tells you the model's *loss* went down — it doesn't tell you whether the generated answers are actually *good*. This notebook closes that gap by:
1. Reloading the exact same `DriveVLMT5` architecture from the training notebook
2. Loading the **best saved checkpoint** back into it
3. Running **autoregressive generation** (`model.generate`) on held-out test questions
4. Scoring the generated captions against ground truth using the **pycocoevalcap** toolkit — the same library used to benchmark image-captioning models on the MS-COCO leaderboard

### Step 1: Import Libraries

Notice the two evaluation-specific imports that didn't appear in the training notebook:
- **`pycocotools.coco.COCO`** — parses annotation files in the COCO JSON format (here, repurposed to hold ground-truth driving-scene answers instead of image captions)
- **`pycocoevalcap.eval.COCOEvalCap`** — computes the actual captioning metrics (BLEU-1 through BLEU-4, METEOR, ROUGE-L, CIDEr) by comparing generated captions to references

Everything else (`torch`, `transformers`, `peft`, `torchvision`) mirrors the training notebook, since we need to reconstruct the *identical* model architecture before we can load its weights.

In [1]:
import os
import json
import torch
import random
import pandas as pd
import torch.nn as nn
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from collections import namedtuple
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, LoftQConfig
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.io import read_image
from torchvision import transforms
from torchvision.models import vit_b_32

/home/exouser/.conda/envs/EM-VLM4AD/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Select Device

Same pattern as training — prefer GPU, fall back to CPU. Generation (autoregressive decoding, token-by-token) is even more compute-intensive than a single training forward pass, so a GPU is strongly recommended here too.

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running evaluation on: {device}")

Running evaluation on: cuda


### Step 3: Evaluation Configuration

The `config` `namedtuple` mirrors the training configuration, but with one crucial addition: **`model_name`**. This must exactly match the **timestamped folder name** that `train_T5_Base.ipynb` created (e.g. `'20260703-164941'`) so that this notebook loads weights from the correct training run.

| Field | Meaning |
|---|---|
| `batch_size` | How many test examples to generate answers for at once |
| `gpa_hidden_size` | Must match the value used during training (the shape of the Gated Pooling Attention layers) |
| `model_name` | Timestamped experiment folder to load the checkpoint from |
| `lm` | Which T5 backbone was trained (`'T5-Base'` or `'T5-Large'`) and must match training exactly |

`VIT_HIDDEN_STATE` and `VIT_SEQ_LENGTH` are the same ViT patch-embedding constants from training, since the vision half of the model is being rebuilt identically.

In [3]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
Config = namedtuple('Instance', ['batch_size', 'gpa_hidden_size', 'model_name', 'lm'])

config = Config(
    batch_size = 12,
    gpa_hidden_size = 128,
    model_name = '20260703-164941',  
    lm = 'T5-Base'
)

VIT_HIDDEN_STATE = 768
VIT_SEQ_LENGTH = 49

### Step 4: Dataset Definition (Evaluation Variant)

This `MultiFrameDataset` is almost identical to the one in `train_T5_Base.ipynb`, where **`collate_fn_test`** additionally returns the raw `img_paths`.

Why the extra return value matters: during evaluation we need to know *which specific image/question* each generated answer corresponds to, so we can match it back to the correct `image_id` in the COCO-style ground-truth annotation file. Training didn't need this because it only needed a loss value, not a traceable mapping back to individual examples.

In [4]:
# ==========================================
# 2. DATASET DEFINITION 
# ==========================================
class MultiFrameDataset(Dataset):
    def __init__(self, input_file, tokenizer, transform=None):
        with open(input_file) as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        qa, img_path = self.data[idx]
        img_paths = list(img_path.values())
        q_text, a_text = qa['Q'], qa['A']
        q_text = f"Question: {q_text} Answer:"

        # KEEP ON CPU HERE to avoid Multiprocessing crashes
        imgs = [self.transform(read_image(p).float()) for p in img_paths]
        imgs = torch.stack(imgs, dim=0)

        return q_text, imgs, a_text, sorted(list(img_path.values()))

    def collate_fn_test(self, batch):
        q_texts, imgs, a_texts, img_paths = zip(*batch)
        imgs = torch.stack(list(imgs), dim=0)
        img_paths = list(img_paths)
        
        # Dynamic padding to prevent sequence length overflow
        encodings = self.tokenizer(q_texts, padding='longest', truncation=True, max_length=512, return_tensors="pt").input_ids
        labels = self.tokenizer(a_texts, padding='longest', truncation=True, max_length=512, return_tensors='pt').input_ids

        return q_texts, encodings, imgs, labels, img_paths


### Step 5: Model Architecture (Identical to Training)

This is a copy of the `DriveVLMT5` / `MultiViewProcessor` classes from `train_T5_Base.ipynb` (frozen ViT → Gated Pooling Attention → fused embeddings → T5). 

One method becomes essential here that wasn't emphasized during training: **`generate()`**. Unlike `forward()` (which computes a loss given known `labels`), `generate()` performs true autoregressive decoding:
1. Builds the same fused text+image embedding sequence as `forward`
2. Seeds the decoder with a single `decoder_start_token_id`
3. Repeatedly predicts the next token and feeds it back in, until an end-of-sequence token or `max_length=512` is reached

This is exactly how the model will be used at inference time in the real world — it has no access to the "correct" answer, only the question and images.

In [5]:
# ==========================================
# 3. MODEL DEFINITION
# ==========================================
def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"Trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}")

class DriveVLMT5(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.lm == 'T5-Base':
            self.model = T5ForConditionalGeneration.from_pretrained('google-t5/t5-base')
        else:
            self.model = T5ForConditionalGeneration.from_pretrained('google-t5/t5-large')
            loftq_config = LoftQConfig(loftq_bits=8)
            lora_config = LoraConfig(r=64, lora_alpha=32, loftq_config=loftq_config, lora_dropout=0.05, bias='none', target_modules=['q', 'v'])
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.d_model
        print('Trainable Parameters for LM model:')
        print_trainable_parameters(self.model)

        self.mvp = self.MultiViewProcessor(config.gpa_hidden_size, hidden_size, config.lm, freeze=True)

    class MultiViewProcessor(nn.Module):
        def __init__(self, gpa_hidden_size, hidden_size, lm, freeze=False):
            super().__init__()
            self.img_model = vit_b_32(weights='DEFAULT')
            self.lm = lm
            self.modal_embeddings = nn.Embedding(2, hidden_size)
            self.modal_embeddings.weight.data.normal_(mean=0.0, std=0.02)

            if freeze:
                for param in self.img_model.parameters():
                    param.requires_grad = False

            self.w = nn.Linear(in_features=gpa_hidden_size, out_features=1)
            self.Z = nn.Sequential(nn.Linear(in_features=VIT_HIDDEN_STATE * VIT_SEQ_LENGTH, out_features=gpa_hidden_size, bias=False), nn.Tanh())
            self.G = nn.Sequential(nn.Linear(in_features=VIT_HIDDEN_STATE * VIT_SEQ_LENGTH, out_features=gpa_hidden_size, bias=False), nn.Sigmoid())
            if self.lm != 'T5-Base':
              self.img_projection_layer = nn.Linear(in_features=VIT_HIDDEN_STATE, out_features=hidden_size)

        def gpa(self, img_embeddings):
            gpa_weights = torch.softmax(self.w(self.Z(img_embeddings) * self.G(img_embeddings)), dim=0)
            return torch.sum(gpa_weights * img_embeddings, dim=0)

        def get_img_embedding(self, imgs):
            N = imgs.shape[0]
            merged_embedding = torch.stack([self.img_model._process_input(img) for img in imgs], dim=0)
            batch_class_tokens = self.img_model.class_token.expand(merged_embedding.shape[1], -1, -1).repeat(N, 1, 1, 1)
            merged_embedding = torch.cat([batch_class_tokens, merged_embedding], dim=2)
            merged_embedding += self.img_model.encoder.pos_embedding.repeat(N, 1, 1, 1)
            merged_embedding = merged_embedding[:, :, 1:]
            merged_embedding = torch.stack([self.gpa(embedding.flatten(start_dim=1)).reshape(VIT_SEQ_LENGTH, VIT_HIDDEN_STATE) for embedding in merged_embedding], dim=0)
            if self.lm != 'T5-Base':
              merged_embedding = self.img_projection_layer(merged_embedding)
            merged_embedding += self.modal_embeddings(torch.ones((1, merged_embedding.shape[1]), dtype=torch.int, device=device))
            return merged_embedding

        def forward(self, text_enc, imgs, text_model):
            imgs_embedding = self.get_img_embedding(imgs)
            text_embeddings = text_model.get_input_embeddings()(text_enc)
            text_embeddings += self.modal_embeddings(torch.zeros((1, text_embeddings.shape[1]), dtype=torch.int, device=device))
            return torch.cat([text_embeddings, imgs_embedding], dim=1)

    def forward(self, text_enc, imgs, labels=None):
        merged_embedding = self.mvp(text_enc, imgs, self.model)
        return self.model(inputs_embeds=merged_embedding, labels=labels)

    def generate(self, text_enc, imgs, lidar=None):
        merged_embedding = self.mvp(text_enc, imgs, self.model)
        attention_mask = torch.ones(merged_embedding.shape[:2], dtype=torch.long, device=device)
        decoder_input_ids = torch.ones((merged_embedding.shape[0], 1), dtype=torch.long, device=device)*self.model.config.decoder_start_token_id
        output_ids = self.model.generate(attention_mask=attention_mask, decoder_input_ids=decoder_input_ids, inputs_embeds=merged_embedding, max_length=512, early_stopping=True)
        return output_ids

### Step 6: Generating Predictions — `val_model`

This function is the evaluation-time counterpart to training's `val_model` (loss computation), but here it performs **generation**, not loss calculation:

1. Iterates over the test `DataLoader` in `model.eval()` / `torch.no_grad()` mode
2. Calls `model.generate(...)` under `bfloat16` autocast to produce token IDs
3. Decodes the generated token IDs back into text with `processor.decode(..., skip_special_tokens=True)`
4. Looks up each `(image_path, question)` pair in a pre-built `image_id_dict` to recover the numeric `image_id` the COCO evaluation format requires
5. **Deduplicates** — `ids_answered` is a `set()` that ensures each `image_id` is only scored once, even if the same scene/question happens to appear more than once in the sampled batch
6. Writes all predictions to `predictions.json` in the standard COCO-captioning results format: a list of `{"image_id": ..., "caption": ...}` dictionaries

This JSON file is exactly what `COCOEvalCap` expects as input in the next step.

In [6]:
# ==========================================
# 4. EVALUATION & METRICS
# ==========================================
def val_model(dloader):
    model.eval()
    ids_answered = set()
    test_data = []

    with torch.no_grad():
        for idx, (q_texts, encodings, imgs, labels, img_paths) in tqdm(enumerate(dloader), total=len(dloader), desc="Evaluating Subsets"):
            
            # 1. Move to device here!
            encodings = encodings.to(device)
            imgs = imgs.to(device)

            # 2. Generate with bfloat16 to avoid NaNs
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                outputs = model.generate(encodings, imgs)

            # Get text outputs
            text_outputs = [processor.decode(output, skip_special_tokens=True) for output in outputs]

            for image_path, q_text, text_output in zip(img_paths, q_texts, text_outputs):
                img_key = image_path[0]

                if image_id_dict[img_key + ' ' + q_text][0] in ids_answered:
                    continue

                ids_answered.add(image_id_dict[img_key + ' ' + q_text][0])
                test_data.append({'image_id': image_id_dict[img_key + ' ' + q_text][0], 'caption': text_output})

    # Save predictions
    out_dir = os.path.join('multi_frame_results', config.model_name)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, 'predictions.json'), 'w') as f:
        json.dump(test_data, f)



### Step 7: Saving Metrics — `save_experiment`

Once `COCOEvalCap` has computed its scores, this helper simply loops over `coco_eval.eval` — a dictionary of `{metric_name: score}` — prints each one, and writes them all into a tidy `metrics.csv` for later comparison across experiments. Typical keys you'll see include:

| Metric | What it measures |
|---|---|
| `Bleu_1`–`Bleu_4` | N-gram precision overlap between generated and reference text (higher-order BLEU penalizes longer wrong phrases more) |
| `METEOR` | N-gram overlap with stemming/synonym matching — more forgiving of paraphrasing than BLEU |
| `ROUGE_L` | Longest common subsequence overlap — rewards preserving sentence structure |
| `CIDEr` | Consensus-based metric originally designed for image captioning; weights n-grams by how *distinctive* they are across the reference set, making it good at rewarding specific, informative answers over generic ones |

In [7]:
def save_experiment():
    trial_dict = {}
    for metric, score in coco_eval.eval.items():
        trial_dict[metric] = [score]
        print(f"{metric}: {score:.4f}") # Print to screen as well

    trial_dict = pd.DataFrame(trial_dict)
    trial_dict.to_csv(os.path.join('multi_frame_results', config.model_name, 'metrics.csv'), index=False, header=True)
    print("Metrics saved to CSV successfully!")


### Step 8: Putting It All Together — Load, Generate, Score

This final cell runs the complete evaluation pipeline:

1. **Rebuild and load the model** — `DriveVLMT5(config)` reconstructs the architecture, then the checkpoint is loaded from `multi_frame_results/<model_name>/best_model.pth`.
   - ⚠️ **The `DataParallel` prefix fix**: If the model was trained across multiple GPUs with `nn.DataParallel`, every saved parameter name is prefixed with `module.` (e.g. `module.mvp.w.weight`). Since evaluation typically runs on a single GPU/CPU *without* `DataParallel`, those prefixes would cause `load_state_dict` to fail with "unexpected key" errors. The `clean_state_dict` loop strips the `module.` prefix so the checkpoint loads cleanly regardless of how many GPUs it was trained on.
2. **Prepare the test dataset** using the same `224×224` resize/normalize transform as training.
3. **Workshop-sized subset** — just like the training notebook capped the training set to 1,500 examples, this notebook evaluates on a **random subset of 100 test examples** rather than the full test split, keeping the exercise fast. Remove the subsetting for a full, publication-grade evaluation.
4. **Run inference** with `val_model(test_dloader)`, producing `predictions.json`.
5. **Run COCO evaluation**:
   - `COCO(annotation_file)` loads the ground-truth answers in COCO format
   - `coco.loadRes(results_file)` loads our generated predictions
   - `coco_eval.params['image_id'] = coco_result.getImgIds()` is an important line — it **restricts scoring to only the 100 images we actually evaluated**, rather than expecting scores for the entire test set (which would otherwise cause errors or misleadingly low scores from "missing" predictions)
   - `coco_eval.evaluate()` computes BLEU/METEOR/ROUGE-L/CIDEr
6. **Save results** to `metrics.csv` via `save_experiment()`

In [9]:
# ==========================================
# 5. EXECUTION SCRIPT
# ==========================================
print("Loading model and processor...")
model = DriveVLMT5(config).to(device)
processor = T5Tokenizer.from_pretrained('google-t5/t5-base' if config.lm == 'T5-Base' else 'google-t5/t5-large')
processor.add_tokens('<')

# Multi-GPU Weight Loading Fix
checkpoint_path = os.path.join('multi_frame_results', config.model_name, 'best_model.pth')
print(f"Loading weights from {checkpoint_path}...")
state_dict = torch.load(checkpoint_path, map_location=device)
clean_state_dict = {}
for k, v in state_dict.items():
    clean_key = k.replace('module.', '') # Clean out DataParallel prefix
    clean_state_dict[clean_key] = v
model.load_state_dict(clean_state_dict)

print("Preparing Dataset...")
test_dset = MultiFrameDataset(
    input_file=os.path.join('data', 'multi_frame', 'multi_frame_test.json'),
    tokenizer=processor,
    transform=transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Normalize((127.5, 127.5, 127.5), (127.5, 127.5, 127.5))
    ])
)

# --- WORKSHOP SUBSET LOGIC (TEST ON 100 SAMPLES) ---
print(f"Original Test Set Size: {len(test_dset)}")
subset_indices = random.sample(range(len(test_dset)), 100)
test_dset_demo = Subset(test_dset, subset_indices)
print(f"Testing on a random subset of: {len(test_dset_demo)} samples")

# Note: drop_last=False so it evaluates exactly 100 images
test_dloader = DataLoader(test_dset_demo, shuffle=False, batch_size=config.batch_size, num_workers=4, drop_last=False, collate_fn=test_dset.collate_fn_test)

with open(os.path.join('data', 'multi_frame', 'image_id.json')) as f:
    image_id_dict = json.load(f)

# 1. Run inference on the 100 samples
val_model(test_dloader)

# 2. Run COCO Evaluation
print("\nStarting COCO Evaluation Metrics Analysis...")
annotation_file = os.path.join('data', 'multi_frame', 'multi_frame_test_coco.json')
results_file = os.path.join('multi_frame_results', config.model_name, 'predictions.json')

coco = COCO(annotation_file)
coco_result = coco.loadRes(results_file)

coco_eval = COCOEvalCap(coco, coco_result)

# This line restricts the COCO grading specifically to the 100 images you evaluated!
coco_eval.params['image_id'] = coco_result.getImgIds()

# Run the grader
coco_eval.evaluate()

# 3. Save and display results
save_experiment()

Loading model and processor...
Trainable Parameters for LM model:
Trainable params: 222903552 || all params: 222903552 || trainable%: 100.00


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading weights from multi_frame_results/20260703-164941/best_model.pth...
Preparing Dataset...
Original Test Set Size: 16817
Testing on a random subset of: 100 samples


Evaluating Subsets:   0%|          | 0/9 [00:00<?, ?it/s]/home/exouser/.conda/envs/EM-VLM4AD/lib/python3.12/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at /opt/conda/conda-bld/pytorch_1708025824022/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
/home/exouser/.conda/envs/EM-VLM4AD/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:453: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
Evaluating Subsets: 100%|██████████| 9/9 [00:05<00:00,  1.56it/s]
PTBTokenizer tokenized 1169 tokens at 22534.77 tokens per second.



Starting COCO Evaluation Metrics Analysis...
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
tokenization...


PTBTokenizer tokenized 637 tokens at 12100.01 tokens per second.


setting up scorers...
computing Bleu score...
{'testlen': 414, 'reflen': 867, 'guess': [414, 314, 265, 233], 'correct': [231, 133, 91, 55]}
ratio: 0.4775086505184804
Bleu_1: 0.187
Bleu_2: 0.163
Bleu_3: 0.145
Bleu_4: 0.125
computing METEOR score...
METEOR: 0.131
computing Rouge score...
ROUGE_L: 0.569
computing CIDEr score...
CIDEr: 2.228
Bleu_1: 0.1868
Bleu_2: 0.1628
Bleu_3: 0.1450
Bleu_4: 0.1246
METEOR: 0.1309
ROUGE_L: 0.5687
CIDEr: 2.2284
Metrics saved to CSV successfully!


## ✅ Recap
You reloaded a trained checkpoint, generated free-form answers with autoregressive decoding, and scored them against ground truth using standard image-captioning metrics repurposed for driving-scene explanations.

### Before you go, think about the following questions:
- Why can't we simply reuse the training loss to judge answer quality — why do we need separate metrics like BLEU/CIDEr?
- The evaluation subset here is only 100 examples. How much would you trust a CIDEr score computed on such a small sample, and why?
- CIDEr rewards *distinctive* phrasing more than generic phrasing. Can you think of a scenario where a driving explanation model might get a low CIDEr score despite giving a *safe and correct* answer?
- If BLEU and CIDEr disagreed on which of two model checkpoints was "better," which metric would you trust more for a driving-explanation task, and why?

## ⏱️ Estimated Time: 20–40 minutes (workshop subset)